<a href="https://colab.research.google.com/github/elkins/synth-nmr/blob/master/docs/tutorials/advanced_observables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced NMR Observables: J-Couplings, NOEs, and RDCs

In this tutorial, we will dive deeper into three crucial NMR observables that provide distinct structural and dynamic information:
- **J-Couplings**: Scalar couplings that depend on backbone dihedral angles.
- **NOEs**: Nuclear Overhauser Effects providing short-range distance restraints.
- **RDCs**: Residual Dipolar Couplings yielding long-range orientational information relative to an alignment tensor.

## 1. Setup Environment & Load Structure

First, we'll install `synth-nmr` and load the Ubiquitin structure (1D3Z) using `biotite`.

In [ ]:
!pip install -q synth-nmr biotite matplotlib

!wget -q https://files.rcsb.org/download/1D3Z.pdb -O ubiquitin.pdb

import biotite.structure.io as strucio

ensemble = strucio.load_structure("ubiquitin.pdb")
structure = ensemble[0]

## 2. J-Couplings: Probing Dihedral Angles

Scalar couplings ($^3J$) are mediated through chemical bonds. The $^3J_{HN-H\alpha}$ coupling strongly correlates with the backbone $\phi$ dihedral angle via the **Karplus equation**.

By predicting these couplings, we can validate the secondary structure of our models. Let's calculate them for our Ubiquitin structure.

In [ ]:
from synth_nmr.j_coupling import calculate_hn_ha_coupling
import matplotlib.pyplot as plt

j_couplings = calculate_hn_ha_coupling(structure)

# Extract couplings for Chain A
chain_a_couplings = j_couplings.get('A', {})
residues = list(chain_a_couplings.keys())
j_vals = list(chain_a_couplings.values())

plt.figure(figsize=(10, 4))
plt.bar(residues, j_vals, color='coral', alpha=0.8)
plt.xlabel("Residue Number")
plt.ylabel("³J(HN-HA) Coupling (Hz)")
plt.title("Predicted Scalar Couplings across Ubiquitin Backbone")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

Values around 4 Hz typically indicate $\alpha$-helical regions, whereas larger values (~9 Hz) often correspond to $\beta$-sheets.

## 3. NOEs: Short-Range Distance Restraints

Nuclear Overhauser Effects (NOEs) provide distance information between protons. The intensity of an NOE signal is proportional to $r^{-6}$, meaning it drops off extremely rapidly beyond 5.0-6.0 Å.

We can calculate synthetic NOEs by finding inter-proton pairs within a cutoff distance. We can also skip intra-residue connections since they're less useful for defining the global fold.

In [ ]:
from synth_nmr.nmr import calculate_synthetic_noes

# Calculate NOEs with a 5.0 Å cutoff, excluding intra-residue pairs
noes = calculate_synthetic_noes(structure, cutoff=5.0, buffer=0.5, exclude_intra_residue=True)

print(f"Generated {len(noes)} inter-residue synthetic NOE distance restraints.\n")

print("Sample NOE Restraints:")
for noe in noes[:5]:
    print(f"  {noe['res_name_1']}{noe['index_1']} {noe['atom_name_1']} <--> "
          f"{noe['res_name_2']}{noe['index_2']} {noe['atom_name_2']}: "
          f"Distance {noe['distance']:.2f} Å (Upper Limit: {noe['upper_limit']:.2f} Å)")

## 4. RDCs: Long-Range Orientational Restraints

Residual Dipolar Couplings (RDCs) measure the angle of a bond vector (like N-H) relative to a global alignment tensor. While NOEs give short-range distances, RDCs provide global orientational information, which makes them highly complementary.

We use two parameters to define the alignment tensor's magnitude and asymmetry:
- **Da**: The axial component of the alignment tensor.
- **R**: The rhombicity, defining deviation from axial symmetry.

In [ ]:
from synth_nmr.rdc import calculate_rdcs

# Using typical values for Da (Hz) and Rhombicity (R)
Da = 10.0  
rhombicity = 0.2

rdcs = calculate_rdcs(structure, Da=Da, R=rhombicity)

print(f"Calculated {len(rdcs)} backbone N-H RDCs.\n")
print("Sample RDC Values (Hz):")
sample_rdcs = list(rdcs.items())[:5]
for res_id, rdc_val in sample_rdcs:
    print(f"  Residue {res_id}: {rdc_val} Hz")

## Summary

By combining **Chemical Shifts** (secondary structure), **Relaxation Rates** (dynamics), **J-Couplings** (dihedrals), **NOEs** (distances), and **RDCs** (orientations), `synth-nmr` allows you to construct a comprehensive profile of how your theoretical 3D model would behave in an NMR magnet!